In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ================== CONFIG ==================

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ASTRUSDT"]
MAIN_DIR = "crypto_data"  # same main folder used in the data-collection script

# Features & target (same as in your notebook)
DF_FEATURES = [
    "hour", "dayofweek", "day",   # time features
    "open", "high", "low", "close", "volume",
    "close_lag_1", "volume_lag_1"
]
TARGET = "target_next_close"

TIME_COLS = ["hour", "dayofweek", "day"]


# ================== HELPERS ==================

def ensure_time_and_lags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make sure hour/dayofweek/day and 1-step lag features exist,
    even if they were not created in the collection script.
    """
    # open_time must be datetime
    if not pd.api.types.is_datetime64_any_dtype(df["open_time"]):
        df["open_time"] = pd.to_datetime(df["open_time"])

    # Time features
    if "hour" not in df.columns:
        df["hour"] = df["open_time"].dt.hour
    if "dayofweek" not in df.columns:
        df["dayofweek"] = df["open_time"].dt.dayofweek
    if "day" not in df.columns:
        df["day"] = df["open_time"].dt.day

    # Lag features (1-step)
    if "close_lag_1" not in df.columns:
        df["close_lag_1"] = df["close"].shift(1)
    if "volume_lag_1" not in df.columns:
        df["volume_lag_1"] = df["volume"].shift(1)

    return df


def preprocess_symbol(symbol: str):
    """
    For a single symbol:
      - Load CSV
      - Ensure time/lag features
      - Select DF_FEATURES + TARGET
      - Train-test split
      - Scale features (except time cols) and target
      - Print shapes
    """
    print("\n================================")
    print(f"Processing {symbol}")
    print("================================")

    # Path to CSV from data collection script
    symbol_dir = os.path.join(MAIN_DIR, symbol)
    csv_path = os.path.join(symbol_dir, f"{symbol}_ML_ready.csv")

    if not os.path.exists(csv_path):
        print(f"❌ CSV not found for {symbol}: {csv_path}")
        return

    # 1️⃣ Load data
    df = pd.read_csv(csv_path)

    # 2️⃣ Ensure time features + lag features
    df = ensure_time_and_lags(df)

    # 3️⃣ Drop rows where any of the selected features or target is NaN
    if TARGET not in df.columns:
        raise ValueError(f"'{TARGET}' column not found in {symbol} dataset. "
                         "Make sure the collection script created 'target_next_close'.")

    df_model = df.dropna(subset=DF_FEATURES + [TARGET]).reset_index(drop=True)

    # 4️⃣ Split into X (features) and y (target)
    X = df_model[DF_FEATURES].copy()
    y = df_model[TARGET].copy()

    # 5️⃣ Train-test split (time-series aware: no shuffle)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        shuffle=False,
        random_state=0  # doesn't matter when shuffle=False, but fine
    )

    # 6️⃣ Scale features (except time columns)
    scale_cols = [col for col in DF_FEATURES if col not in TIME_COLS]

    scaler_X = MinMaxScaler()
    scaler_X.fit(X_train[scale_cols])

    # Create copies to avoid changing original X_train/X_test
    X_train_final = X_train.copy()
    X_test_final = X_test.copy()

    X_train_final[scale_cols] = scaler_X.transform(X_train[scale_cols])
    X_test_final[scale_cols] = scaler_X.transform(X_test[scale_cols])

    # 7️⃣ Scale target
    scaler_y = MinMaxScaler()
    y_train_arr = y_train.values.reshape(-1, 1)
    y_test_arr = y_test.values.reshape(-1, 1)

    scaler_y.fit(y_train_arr)
    y_train_scaled = scaler_y.transform(y_train_arr)
    y_test_scaled = scaler_y.transform(y_test_arr)

    # 8️⃣ Print shapes (same kind of check as in your notebook)
    print(f"{symbol} ->")
    print("  X_train_final:", X_train_final.shape)
    print("  X_test_final :", X_test_final.shape)
    print("  y_train_scaled:", y_train_scaled.shape)
    print("  y_test_scaled :", y_test_scaled.shape)

    # (Optional) If you want to save processed arrays/scalers, uncomment below:

    # import joblib
    # joblib.dump(scaler_X, os.path.join(symbol_dir, f"{symbol}_scaler_X.pkl"))
    # joblib.dump(scaler_y, os.path.join(symbol_dir, f"{symbol}_scaler_y.pkl"))
    #
    # X_train_final.to_csv(os.path.join(symbol_dir, f"{symbol}_X_train_final.csv"), index=False)
    # X_test_final.to_csv(os.path.join(symbol_dir, f"{symbol}_X_test_final.csv"), index=False)
    #
    # pd.DataFrame(y_train_scaled, columns=[TARGET]).to_csv(
    #     os.path.join(symbol_dir, f"{symbol}_y_train_scaled.csv"), index=False
    # )
    # pd.DataFrame(y_test_scaled, columns=[TARGET]).to_csv(
    #     os.path.join(symbol_dir, f"{symbol}_y_test_scaled.csv"), index=False
    # )

    # Return in case you want to use in a notebook
    return X_train_final, X_test_final, y_train_scaled, y_test_scaled


# ================== MAIN ==================

if __name__ == "__main__":
    for sym in SYMBOLS:
        try:
            preprocess_symbol(sym)
        except Exception as e:
            print(f"❌ Error while processing {sym}: {e}")



Processing BTCUSDT
BTCUSDT ->
  X_train_final: (35033, 10)
  X_test_final : (8759, 10)
  y_train_scaled: (35033, 1)
  y_test_scaled : (8759, 1)

Processing ETHUSDT
ETHUSDT ->
  X_train_final: (35033, 10)
  X_test_final : (8759, 10)
  y_train_scaled: (35033, 1)
  y_test_scaled : (8759, 1)

Processing BNBUSDT
BNBUSDT ->
  X_train_final: (35033, 10)
  X_test_final : (8759, 10)
  y_train_scaled: (35033, 1)
  y_test_scaled : (8759, 1)

Processing XRPUSDT
XRPUSDT ->
  X_train_final: (35033, 10)
  X_test_final : (8759, 10)
  y_train_scaled: (35033, 1)
  y_test_scaled : (8759, 1)

Processing ASTRUSDT
ASTRUSDT ->
  X_train_final: (19900, 10)
  X_test_final : (4976, 10)
  y_train_scaled: (19900, 1)
  y_test_scaled : (4976, 1)
